## **06_multi_head_attention: Many Conversations at Once**

So far, we've built a single, causal self-attention mechanism.  
It's like having **one person** in a meeting responsible for figuring out **all** the relationships between words.  
Grammar, meaning, long-range context — all at once. That's a lot of pressure.

### The Idea: Parallel Conversations

What if, instead of one overworked mechanism, we had **several working in parallel**?

This is **Multi-Head Attention**. We split our embedding dimension `C` into smaller chunks called "heads".  
Each head is its own independent attention mechanism:

+ **Head 1** might become a specialist in syntax (verb-object relationships)
+ **Head 2** might specialize in semantics (related meanings)
+ **Head 3** might track pronoun references
+ ...and so on

The entire process is just: **Split → Attend → Merge**

### Step 1: Splitting `C` into `n_head` and `head_dim`

Let's use the actual dimensions from GPT-2 small:
+ Embedding dimension `C = 768`
+ Number of heads `n_head = 12`
+ Dimension per head: `head_dim = C / n_head = 768 / 12 = 64`

We reshape `(B, T, C)` → `(B, n_head, T, head_dim)` using `.view()` and `.transpose()`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

B, T, C = 1, 4, 768
n_head = 12
head_dim = C // n_head  # 768 // 12 = 64

# Dummy Q, K, V tensors with realistic shapes
q = torch.randn(B, T, C)
k = torch.randn(B, T, C)
v = torch.randn(B, T, C)

print("Original Q shape:", q.shape)

# Step 1.1: view — carve up the last dimension
q_reshaped = q.view(B, T, n_head, head_dim)  # (1, 4, 12, 64)
print("After .view():", q_reshaped.shape)

# Step 1.2: transpose — bring n_head to the front
q_final = q_reshaped.transpose(1, 2)  # (1, 12, 4, 64)
print("After .transpose(1, 2):", q_final.shape)

The key insight: by moving `n_head` to the front, PyTorch's broadcasting treats it like a **new batch dimension**.  
All our attention math will now happen **12 times at once, independently**.

### Step 2: Run Attention in Parallel

The attention formula is the same — it just operates on tensors with an extra `n_head` dimension.

```
(B, nh, T, hd) @ (B, nh, hd, T) → (B, nh, T, T)    # scores
(B, nh, T, T)  @ (B, nh, T, hd) → (B, nh, T, hd)    # output
```

In [ ]:
# Reshape k and v the same way
k_final = k.view(B, T, n_head, head_dim).transpose(1, 2)  # (1, 12, 4, 64)
v_final = v.view(B, T, n_head, head_dim).transpose(1, 2)  # (1, 12, 4, 64)

# Attention calculation — happening 12 times at once!
scaled_scores = (q_final @ k_final.transpose(-2, -1)) / math.sqrt(head_dim)
print("Scaled scores shape:", scaled_scores.shape)  # (1, 12, 4, 4)

# (We would apply the causal mask here)

attention_weights = F.softmax(scaled_scores, dim=-1)

output_per_head = attention_weights @ v_final
print("Output per head shape:", output_per_head.shape)  # (1, 12, 4, 64)

Each of our 12 specialists has done its job.  
We have a 64-dimensional output vector for each of our 4 tokens, from each of our 12 heads.

### Step 3: Merging the Heads

Time to bring the team back together.  
We reverse the reshape: concatenate heads back into a single `C`-dimensional vector,  
then pass through a final linear projection (`c_proj`).

In [ ]:
# 1. Transpose and reshape to merge heads
# (B, nh, T, hd) -> (B, T, nh, hd) -> (B, T, C)
merged_output = output_per_head.transpose(1, 2).contiguous()
# .contiguous() is needed because transpose can mess with memory layout
merged_output = merged_output.view(B, T, C)
print("Merged output shape:", merged_output.shape)

# 2. Final projection layer
c_proj = nn.Linear(C, C)
final_output = c_proj(merged_output)
print("Final output shape:", final_output.shape)

We're back to `(B, T, C)`. Each token's vector now contains the combined, context-aware information from all 12 attention heads.

| Component | Shape Transformation | Purpose |
| :--- | :--- | :--- |
| **Split Heads** | `(B, T, C) → (B, nh, T, hd)` | Prepare for parallel computation |
| **Attention** | `(B, nh, T, hd) → (B, nh, T, hd)` | Each head computes context independently |
| **Merge Heads** | `(B, nh, T, hd) → (B, T, C)` | Combine insights from all heads |
| **Final Projection** | `(B, T, C) → (B, T, C)` | Mix the combined information |

### The Full `CausalSelfAttention` Module

Combining everything: the fused projection, causal mask, multi-head splitting, and final projection.

In [ ]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    n_embd: int = 768
    n_head: int = 12
    block_size: int = 1024

class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=True)

        self.register_buffer(
            "bias",
            torch.tril(torch.ones(config.block_size, config.block_size))
            .view(1, 1, config.block_size, config.block_size)
        )

    def forward(self, x):
        B, T, C = x.size()

        # 1. Get QKV and split into heads
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)

        # 2. Run causal self-attention on each head
        att = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = att @ v

        # 3. Merge heads and project
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)

        return y

# Test
config = GPTConfig(n_embd=768, n_head=12, block_size=1024)
attn = CausalSelfAttention(config)
x = torch.randn(1, 4, 768)
out = attn(x)
print("Input shape:", x.shape)
print("Output shape:", out.shape)

Every single line in this module should now be crystal clear.  
We have built the most complex and important component of the Transformer from the ground up.

That was the **communication** layer — the big group meeting where tokens share information.  
Now it's time for the **thinking** layer: after the meeting, each token goes back to its desk to process what it heard.  
This is the **MLP**.